# Optimizing Basket Selection via Independence

In [ ]:
import pandas as pd
import numpy as np
from csm.data import download_data
from csm.universe import SP500WikipediaUniverse
from csm.config import Config

cfg = Config(
    use_cache=True
)

tickers = SP500WikipediaUniverse.load(cfg)

In [5]:
data = download_data(cfg, tickers)

In [7]:
data.Close.dropna(how='all', axis=1, inplace=True)
data.Close.drop(columns=['SPY'], inplace=True, errors='ignore')

In [8]:
returns = data.Close.apply(lambda x: np.log1p(x.pct_change())).dropna(how='all', axis=0)

In [9]:
corr = returns.corr()

In [10]:
from scipy.spatial.distance import squareform
dist = 1 - corr.abs()      # ignore sign
dist = dist.fillna(1)

X_dist = dist.values.copy()
np.fill_diagonal(X_dist, 0.0)
X_dist = squareform(X_dist)

In [11]:
import kmedoids

D = dist.values.copy()
np.fill_diagonal(D, 0.0)

km = kmedoids.KMedoids(200, method='fasterpam', random_state=42)
result = km.fit(D)
labels = result.labels_

# medoid_indices are the actual "most representative / most independent" picks
medoid_idx = result.medoid_indices_
tickers = dist.columns.to_numpy()
independent_basket = tickers[medoid_idx]

print(independent_basket)

['FRT' 'IT' 'LULU' 'SW' 'LITE' 'BALL' 'RMD' 'CVS' 'TTWO' 'IFF' 'ADM'
 'FITB' 'KR' 'CDW' 'ROL' 'MCO' 'XYZ' 'AMT' 'AJG' 'ADP' 'SBUX' 'WYNN' 'PSA'
 'GD' 'CASY' 'MA' 'VRTX' 'WSM' 'PH' 'NOW' 'MAR' 'HCA' 'VST' 'EW' 'ABBV'
 'LMT' 'KHC' 'INCY' 'GM' 'ULTA' 'ODFL' 'BAX' 'BF-B' 'FDXF' 'ELV' 'DLR' 'A'
 'GIS' 'MNST' 'STZ' 'NSC' 'NEM' 'HOOD' 'TKO' 'FOX' 'BBY' 'DVA' 'VTRS'
 'BAC' 'CB' 'CTVA' 'SOLV' 'SYK' 'RSG' 'LLY' 'APP' 'DAL' 'KVUE' 'TMUS' 'HD'
 'OTIS' 'MRNA' 'PHM' 'PODD' 'FISV' 'BKNG' 'ITW' 'CVNA' 'DG' 'ALGN' 'KDP'
 'ZBRA' 'VLO' 'KKR' 'MSI' 'CHTR' 'COP' 'STLD' 'HPQ' 'ROST' 'RCL' 'VTR'
 'TPR' 'WST' 'TGT' 'PANW' 'CF' 'EL' 'OKE' 'GRMN' 'IQV' 'AMGN' 'AKAM'
 'ORCL' 'NKE' 'BDX' 'CLX' 'ADI' 'MRK' 'DECK' 'LYB' 'FSLR' 'CTSH' 'AMCR'
 'EA' 'FFIV' 'ETN' 'GILD' 'CSCO' 'TSCO' 'PSKY' 'EXPD' 'IBKR' 'ECHO' 'VLTO'
 'HONA' 'T' 'GEN' 'TAP' 'PLTR' 'HAS' 'AXON' 'REGN' 'GEHC' 'DLTR' 'HSIC'
 'DELL' 'LRCX' 'GOOG' 'HRL' 'TROW' 'ORLY' 'WDC' 'PG' 'TTD' 'LNT' 'LH'
 'YUM' 'JNJ' 'INTC' 'NTAP' 'DIS' 'ANET' 'NFLX' 'GDDY' 'PCG' 'A

In [ ]:
### === TODO === ###
### Read up on KMedoids
### Come up with a way to measure independence
###     is this just saying "these are the most independent" => can they be at all?